**What are you trying to do in this notebook?**

My job is to determine the most optimal way to craft this year’s Christmas card, by selecting the most efficient path of both moving the robotic arm and changing the print color to craft this year’s image. Each link of the printer arm can be moved independently each step, but I'll also need to account for the time needed to change the printing color.

**Why are you trying it?**

To create a sequence of arm configurations covering every point on the image that minimizes the total movement of the arm and also the change in color from point to point.

The position of the arm is the sum of these displacement vectors and indicates the location of the tip of the arm. The base of the arm (the origin of the first vector) is at , which is the midpoint of the image.

The arm can be reconfigured step-by-step by rotating any or all of the links by 1 unit, incurring a total reconfiguration cost equal to the square root of the number of links changed. Additionally, it incurs a color cost equal to the sum of the absolute differences in the color components from one step to the next and multiplied by a scaling factor of 3.0.

The task is to find a sequence of configurations with positions at every point in the solution image having a minimal cost.

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import matplotlib.pyplot as plt

from tqdm import tqdm
from functools import *
from itertools import *

In [ ]:
df = pd.read_csv('/kaggle/input/santa-2022/image.csv')
df.head()

In [ ]:
plt.figure(figsize=(10,10))
plt.scatter(x=df['x'], y=df['y'], c=df[['r','g','b']].apply(lambda x: (x['r'], x['g'], x['b']), axis=1))
plt.show()

In [ ]:
def get_position(config):
    return reduce(lambda p, q: (p[0] + q[0], p[1] + q[1]), config, (0, 0))

In [ ]:
def rotate_link(vector, direction):
    x, y = vector
    if direction == 1:  
        if y >= x and y > -x:
            x -= 1
        elif y > x and y <= -x:
            y -= 1
        elif y <= x and y < -x:
            x += 1
        else:
            y += 1
    elif direction == -1:  
        if y > x and y >= -x:
            x += 1
        elif y >= x and y < -x:
            y += 1
        elif y < x and y <= -x:
            x -= 1
        else:
            y -= 1
    return (x, y)

In [ ]:
def rotate(config, i, direction):
    config = config.copy()
    config[i] = rotate_link(config[i], direction)
    return config

In [ ]:
def cartesian_to_array(x, y, shape):
    m, n = shape[:2]
    i = (n - 1) // 2 - y
    j = (n - 1) // 2 + x
    if i < 0 or i >= m or j < 0 or j >= n:
        raise ValueError("Coordinates not within given dimensions.")
    return i, j

In [ ]:
def reconfiguration_cost(from_config, to_config):
    nlinks = len(from_config)
    diffs = np.abs(np.asarray(from_config) - np.asarray(to_config)).sum(axis=1)
    return np.sqrt(diffs.sum())

In [ ]:
def color_cost(from_position, to_position, image, color_scale=3.0):
    return np.abs(image[to_position] - image[from_position]).sum() * color_scale

In [ ]:
def step_cost(from_config, to_config, image):
    from_position = cartesian_to_array(*get_position(from_config), image.shape)
    to_position = cartesian_to_array(*get_position(to_config), image.shape)
    return (reconfiguration_cost(from_config, to_config) +
            color_cost(from_position, to_position, image))

In [ ]:
def df_to_image(df):
    side = int(len(df) ** 0.5)  
    return df.set_index(['x', 'y']).to_numpy().reshape(side, side, -1)
image = df_to_image(df)

In [ ]:
def compress_path(path):
    r = [[] for _ in range(8)]
    for p in path:
        for i in range(8):
            if len(r[i]) == 0 or r[i][-1] != p[i]:
                r[i].append(p[i])
    mx = max([len(x) for x in r])
    
    for rr in r:
        while len(rr) < mx:
            rr.append(rr[-1])
    r = list(zip(*r))
    for i in range(len(r)):
        r[i] = list(r[i])
    return r

In [ ]:
def get_direction(u, v):
    """Returns the sign of the angle from u to v."""
    direction = np.sign(np.cross(u, v))
    if direction == 0 and np.dot(u, v) < 0:
        direction = 1
    return direction

In [ ]:
def get_path_to_point(config, point):
    """Find a path of configurations to `point` starting at `config`."""
    path = [config]

    for i in range(len(config)):
        link = config[i]
        base = get_position(config[:i])
        relbase = (point[0] - base[0], point[1] - base[1])
        position = get_position(config[:i+1])
        relpos = (point[0] - position[0], point[1] - position[1])
        radius = reduce(lambda r, link: r + max(abs(link[0]), abs(link[1])), config[i+1:], 0)
        
        if radius == 1 and relpos == (0, 0):
            config = rotate(config, i, 1)
            if get_position(config) == point:  
                path.append(config)
                break
            else:
                continue
        while np.max(np.abs(relpos)) > radius:
            direction = get_direction(link, relbase)
            config = rotate(config, i, direction)
            path.append(config)
            link = config[i]
            base = get_position(config[:i])
            relbase = (point[0] - base[0], point[1] - base[1])
            position = get_position(config[:i+1])
            relpos = (point[0] - position[0], point[1] - position[1])
            radius = reduce(lambda r, link: r + max(abs(link[0]), abs(link[1])), config[i+1:], 0)
    assert get_position(path[-1]) == point
    path = compress_path(path) 
    return path

In [ ]:
def get_path_to_configuration(from_config, to_config):
    path = [from_config]
    config = from_config.copy()
    while config != to_config:
        for i in range(len(config)):
            config = rotate(config, i, get_direction(config[i], to_config[i]))
        path.append(config)
    assert path[-1] == to_config
    return path

In [ ]:
def total_cost(path, image):
    return reduce(lambda cost, pair: cost + step_cost(pair[0], pair[1], image),
                  zip(path[:-1], path[1:]), 0)

In [ ]:
origin = [(64, 0), (-32, 0), (-16, 0), (-8, 0), (-4, 0), (-2, 0), (-1, 0), (-1, 0)]

In [ ]:
l, r = 257, 128
points = []

d = 1
for k in range(2):
    for i in range(r+1):
        points.append((i, 1 * d))

    flag = False
    for i in range(r-1):
        if flag:
            for j in range(l-1):
                points.append((j-r+1, (i+2) * d))
        else:
            for j in range(l, 1, -1):
                points.append((j-r-1, (i+2) * d))
        flag = not flag

    for i in range(r):
        points.append((-r, (r-i) * d))

    for i in range(r-1):
        points.append((-r+i+1, 1 * d))
    
    d *= -1

for i in range(l):
    points.append((i-r, 0))

In [ ]:
visited = set()
path = [origin]
for p in tqdm(points):
    config = path[-1]
    if p not in visited:
        candy_cane_road = get_path_to_point(config, p)[1:]
        if len(candy_cane_road)>0:
            visited |= set([get_position(r) for r in candy_cane_road])
        path.extend(candy_cane_road)
path.extend(get_path_to_configuration(path[-1], origin)[1:])
total_cost(path, image)

In [ ]:
def config_to_string(config):
    return ';'.join([' '.join(map(str, vector)) for vector in config])
submission = pd.Series([config_to_string(config) for config in path], name="configuration")
submission.to_csv('submission.csv', index=False)
submission.head()

**Share your feedback to accomplish the goal!**